In [85]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,accuracy_score ,confusion_matrix

In [86]:
df = pd.read_csv("UNSW_NB15_training-set.csv")

In [87]:
df = df.drop(columns=['attack_cat', 'id'])

In [88]:
df.head()

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,label
0,0.000011,udp,-,INT,2,0,496,0,90909.0902,254,...,1,1,2,0,0,0,1,2,0,0
1,0.000008,udp,-,INT,2,0,1762,0,125000.0003,254,...,1,1,2,0,0,0,1,2,0,0
2,0.000005,udp,-,INT,2,0,1068,0,200000.0051,254,...,1,1,3,0,0,0,1,3,0,0
3,0.000006,udp,-,INT,2,0,900,0,166666.6608,254,...,2,1,3,0,0,0,2,3,0,0
4,0.000010,udp,-,INT,2,0,2126,0,100000.0025,254,...,2,1,3,0,0,0,2,3,0,0


In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82332 entries, 0 to 82331
Data columns (total 43 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   dur                82332 non-null  float64
 1   proto              82332 non-null  object 
 2   service            82332 non-null  object 
 3   state              82332 non-null  object 
 4   spkts              82332 non-null  int64  
 5   dpkts              82332 non-null  int64  
 6   sbytes             82332 non-null  int64  
 7   dbytes             82332 non-null  int64  
 8   rate               82332 non-null  float64
 9   sttl               82332 non-null  int64  
 10  dttl               82332 non-null  int64  
 11  sload              82332 non-null  float64
 12  dload              82332 non-null  float64
 13  sloss              82332 non-null  int64  
 14  dloss              82332 non-null  int64  
 15  sinpkt             82332 non-null  float64
 16  dinpkt             823

In [90]:
df.describe()

,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,...,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,label
count,82332.000000,82332.000000,82332.000000,8.233200e+04,8.233200e+04,8.233200e+04,82332.000000,82332.000000,8.233200e+04,8.233200e+04,...,82332.000000,82332.000000,82332.000000,82332.000000,82332.000000,82332.000000,82332.000000,82332.000000,82332.000000,82332.000000
mean,1.006756,18.666472,17.545936,7.993908e+03,1.323379e+04,8.241089e+04,180.967667,95.713003,6.454902e+07,6.305470e+05,...,4.928898,3.663011,7.456360,0.008284,0.008381,0.129743,6.468360,9.164262,0.011126,0.550600
std,4.710444,133.916353,115.574086,1.716423e+05,1.514715e+05,1.486204e+05,101.513358,116.667722,1.798618e+08,2.393001e+06,...,8.389545,5.915386,11.415191,0.091171,0.092485,0.638683,8.543927,11.121413,0.104891,0.497436
min,0.000000,1.000000,0.000000,2.400000e+01,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,...,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000
25%,0.000008,2.000000,0.000000,1.140000e+02,0.000000e+00,2.860611e+01,62.000000,0.000000,1.120247e+04,0.000000e+00,...,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,2.000000,0.000000,0.000000
50%,0.014138,6.000000,2.000000,5.340000e+02,1.780000e+02,2.650177e+03,254.000000,29.000000,5.770032e+05,2.112951e+03,...,1.000000,1.000000,3.000000,0.000000,0.000000,0.000000,3.000000,5.000000,0.000000,1.000000
75%,0.719360,12.000000,10.000000,1.280000e+03,9.560000e+02,1.111111e+05,254.000000,252.000000,6.514286e+07,1.585808e+04,...,4.000000,3.000000,6.000000,0.000000,0.000000,0.000000,7.000000,11.000000,0.000000,1.000000
max,59.999989,10646.000000,11018.000000,1.435577e+07,1.465753e+07,1.000000e+06,255.000000,253.000000,5.268000e+09,2.082111e+07,...,59.000000,38.000000,63.000000,2.000000,2.000000,16.000000,60.000000,62.000000,1.000000,1.000000


#### traitement de valeurs manquantes

In [91]:
df['service'] = df['service'].replace({"-": np.nan, "nan": np.nan})
mode_service = df['service'].mode()[0]
print(f"Mode de service : {mode_service}")
df['service'] = df['service'].fillna(mode_service)

Mode de service : dns


In [92]:
df.head()

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,label
0,0.000011,udp,dns,INT,2,0,496,0,90909.0902,254,...,1,1,2,0,0,0,1,2,0,0
1,0.000008,udp,dns,INT,2,0,1762,0,125000.0003,254,...,1,1,2,0,0,0,1,2,0,0
2,0.000005,udp,dns,INT,2,0,1068,0,200000.0051,254,...,1,1,3,0,0,0,1,3,0,0
3,0.000006,udp,dns,INT,2,0,900,0,166666.6608,254,...,2,1,3,0,0,0,2,3,0,0
4,0.000010,udp,dns,INT,2,0,2126,0,100000.0025,254,...,2,1,3,0,0,0,2,3,0,0


#### traitement des valeurs aberrantes

In [93]:
def traiter_outliers(df):
    colonnes_numeriques = df.select_dtypes(include=['number']).columns
    
    for col in colonnes_numeriques:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        nb_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        
        if nb_outliers > 0:
            df[col] = df[col].clip(lower=lower, upper=upper)
            print(f"{col} : {nb_outliers} outliers traités")
        else:
            print(f"{col} : aucun outlier, colonne ignorée")
    
    return df

In [94]:
df=traiter_outliers(df)

dur : 5868 outliers traités
spkts : 10196 outliers traités
dpkts : 8907 outliers traités
sbytes : 9270 outliers traités
dbytes : 12308 outliers traités
rate : 6201 outliers traités
sttl : aucun outlier, colonne ignorée
dttl : aucun outlier, colonne ignorée
sload : 6715 outliers traités
dload : 18112 outliers traités
sloss : 5499 outliers traités
dloss : 11272 outliers traités
sinpkt : 5668 outliers traités
dinpkt : 4717 outliers traités
sjit : 6321 outliers traités
djit : 8573 outliers traités
swin : aucun outlier, colonne ignorée
stcpb : aucun outlier, colonne ignorée
dtcpb : aucun outlier, colonne ignorée
dwin : aucun outlier, colonne ignorée
tcprtt : 2020 outliers traités
synack : 2954 outliers traités
ackdat : 2480 outliers traités
smean : 11928 outliers traités
dmean : 9902 outliers traités
trans_depth : 7582 outliers traités
response_body_len : 5657 outliers traités
ct_srv_src : 10093 outliers traités
ct_state_ttl : 1833 outliers traités
ct_dst_ltm : 10479 outliers traités
ct_src

#### Standardisation des données

In [95]:
def standardiser(df):
    # Sélectionner uniquement les colonnes numériques
    colonnes_numeriques = df.select_dtypes(include=['number']).columns.tolist()
    
    # Exclure la colonne cible
    colonnes_numeriques = [col for col in colonnes_numeriques 
                           if col not in ['label', 'attack_cat']]
    scaler = StandardScaler()
    df[colonnes_numeriques] = scaler.fit_transform(df[colonnes_numeriques]) # Appliquer la standardisation
    return df, scaler

In [96]:
df, scaler = standardiser(df)

#### Encodage les variables catégorielles

In [97]:
def encoder_categoriel(df):
    colonnes_cat = df.select_dtypes(include=['object']).columns.tolist()
    print(f"Colonnes catégorielles trouvées : {colonnes_cat}")
    one = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    
    encoded_array = one.fit_transform(df[colonnes_cat])
    
    encoded_df = pd.DataFrame(
        encoded_array,
        columns=one.get_feature_names_out(colonnes_cat),
        index=df.index
    )
    df = df.drop(columns=colonnes_cat) # Supprimer les colonnes originales et ajouter les encodées
    df = pd.concat([df, encoded_df], axis=1)
    
    print(f"{len(colonnes_cat)} colonnes encodées : {colonnes_cat}")
    print(f" Nouvelles dimensions du DataFrame : {df.shape}")
    return df

In [98]:
df = encoder_categoriel(df)

Colonnes catégorielles trouvées : ['proto', 'service', 'state']
3 colonnes encodées : ['proto', 'service', 'state']
 Nouvelles dimensions du DataFrame : (82332, 190)


#### Définir la variable cible

In [99]:
X = df.drop(['label'])
y = df['label']

#### Séparer Train/Test

In [101]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Model Training

In [102]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


#### Model Evaluation

In [103]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred,
      target_names=['Normal', 'Brute Force']))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

      Normal       0.97      0.98      0.97      7418
 Brute Force       0.98      0.97      0.98      9049

    accuracy                           0.97     16467
   macro avg       0.97      0.97      0.97     16467
weighted avg       0.97      0.97      0.97     16467

[[7246  172]
 [ 252 8797]]


In [104]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy :", accuracy * 100, "%")

Accuracy : 97.42515333697699 %
